# Contextual Retrieval：给片段补上模型生成的背景

Contextual Retrieval 是一种给片段补背景的方法：建立索引时，先让模型说明片段在整份资料中的位置，再把“背景 + 原片段”一起编码。它和只补章节标题不同，背景文字由模型生成，但最终回答仍只能引用原片段。

本页做一个轻量候选集教学实验。先把原文切成短片段，用 BM25 为两道问题预筛候选片段，再让 `.env` 中配置的 `glm-4-flash` 根据片段及其所在页的原文生成背景，经过目录页/长度规则校验后，用本地 BGE 比较原片段和“背景 + 原片段”的排名。Notebook 保存的是当前代码的真实 API 运行结果，并完整保留每个候选片段的原始模型输出。结果只代表这批候选片段，不是全库索引收益。


In [1]:
import json
import sys
from pathlib import Path

def find_tutorial_root(start: Path) -> Path:
    for folder in [start, *start.parents, start / "notebook" / "C7 高级 RAG 技巧"]:
        if (folder / "data" / "dataset/manifest.json").is_file() and (folder / "common" / "eval_utils.py").is_file():
            return folder
    raise FileNotFoundError("找不到教程数据目录，请从教程所在目录运行")

TUTORIAL_ROOT = find_tutorial_root(Path.cwd().resolve())
sys.path.insert(0, str(TUTORIAL_ROOT))
from common.eval_utils import emit_tutorial_audit
from common.nontraining_utils import (
    RAG_LLM_MODEL, build_bm25_chunk_search, build_page_dense_search,
    load_annotation, load_pdf_pages, load_query_only, llm_call,
    rank_and_coverage,
)
from common.eval_utils import make_fixed_chunks

CASE_IDS = ["contextual_model_eval_scope", "contextual_cv_three_methods"]
queries = load_query_only(CASE_IDS)
pages = load_pdf_pages()
chunks = make_fixed_chunks(pages, chunk_size=260, overlap=40)
bm25 = build_bm25_chunk_search(chunks)

# 这是候选集教学实验：BM25 只为两道评测问题预筛候选，不是全库索引。
# 生产离线构建应对 chunks 全量生成并缓存背景；本页保存结果不能外推到全书。
candidate_ids = set()
for item in queries:
    candidate_ids.update(hit.chunk_id for hit in bm25(item["query"], top_k=12))
candidate_rows = [
    dict(row, page=int(row["pages"][0]))
    for row in chunks
    if row["chunk_id"] in candidate_ids
]
raw_search = build_page_dense_search(candidate_rows)

contextual_rows = []
raw_generated_contexts = {}
validated_contexts = {}
validation_notes = {}
page_text = {int(row["page"]): str(row["text"]) for row in pages}

def is_directory_page(text):
    # 只依据页文本的目录特征判断，不读取评测标注。
    terms = ("目录", "第1章", "第2章", "第14章", "基本术语", "模型评估与选择")
    return "目录" in text or sum(term in text for term in terms) >= 3

def validate_generated_context(raw, source_page):
    # 模型输出原样留档；最终用于检索的文字由可解释规则校验。
    raw = str(raw or "").strip()
    if not raw:
        raise ValueError("模型返回空背景，终止索引构建；请检查 API 调用，不允许静默回退。")
    if is_directory_page(source_page):
        return ("所在页是目录页；该片段只记录章节/小节的目录位置，不视为正文解释。",
                "目录特征命中；用不引入事实的固定说明替换模型概括。")
    if len(raw) > 120:
        return raw[:120].rstrip("。；") + "。", "超过背景长度边界；截断最终背景，原始输出不变。"
    return raw, "通过页类型、非空和长度规则；未把背景当作答案。"
for row in candidate_rows:
    generated_context = llm_call(
        "你在做 Contextual Retrieval 索引。请根据短 chunk 及其所在页的局部原文，写一句不超过 45 个字的背景，"
        "说明这个 chunk 在所在主题中的位置。不要回答任何用户问题，不要编造页码，"
        "不要加入原文没有的事实。\n所在页局部原文：" + page_text[row["page"]][:700]
        + "\n待加背景的短 chunk：" + str(row["text"]),
        max_tokens=120,
    )
    chunk_id = str(row["chunk_id"])
    raw_generated_contexts[chunk_id] = generated_context
    validated_contexts[chunk_id], validation_notes[chunk_id] = validate_generated_context(
        generated_context, page_text[row["page"]]
    )
    contextual_rows.append({
        "chunk_id": row["chunk_id"],
        "page": row["page"],
        "original_text": str(row["text"]),
        "context": validated_contexts[chunk_id],
        "search_text": validated_contexts[chunk_id] + "\n原始片段：" + str(row["text"]),
        "text": validated_contexts[chunk_id] + "\n原始片段：" + str(row["text"]),
    })
contextual_search = build_page_dense_search(contextual_rows)

records = []
for item in queries:
    question = item["query"]
    before = raw_search(question, top_k=min(8, len(candidate_rows)))
    after = contextual_search(question, top_k=min(8, len(candidate_rows)))
    records.append({
        "case_id": item["id"],
        "query": question,
        "before": before,
        "after": after,
        "model_outputs": {
            "candidate_chunks": [str(row["chunk_id"]) for row in candidate_rows],
            "raw_generated_contexts": raw_generated_contexts,
            "validated_contexts": validated_contexts,
            "validation_notes": validation_notes,
        },
    })

# 结果产生后才读取评估标注并检查页覆盖。
def print_contextual_report(record):
    annotation = record["annotation"]
    before = rank_and_coverage(record["before"], annotation["expected_pages"])
    after = rank_and_coverage(record["after"], annotation["expected_pages"])
    before_chars = sum(len(item.text) for item in record["before"])
    after_chars = sum(len(item.text) for item in record["after"])
    example_context = next(iter(record.get("model_outputs", {}).get("validated_contexts", {}).values()), "")
    print("问题：", record["query"])
    print("改动前：必要页排名", before["first_required_rank"], "，覆盖", before["required_pages_found"], "；改动后：必要页排名", after["first_required_rank"], "，覆盖", after["required_pages_found"])
    print("模型补充的一句背景：", str(example_context)[:120])
    print("资料量：候选", len(candidate_rows), "条；返回", len(record["before"]), "→", len(record["after"]), "条；字符数", before_chars, "→", after_chars, "；背景生成调用", len(raw_generated_contexts), "次")
    before_rank, after_rank = before["first_required_rank"], after["first_required_rank"]
    if before_rank is None and after_rank is not None:
        conclusion = "本次必要页排名改善；仍需在更多问题上验证。"
    elif before_rank is None and after_rank is None:
        conclusion = "本次前后都未命中必要页；没有证据表明背景文字带来收益。"
    elif after_rank is None:
        conclusion = "本次从命中变为未命中；背景文字造成退化。"
    elif after_rank < before_rank:
        conclusion = "本次必要页排名改善；仍需在更多问题上验证。"
    elif before_rank == after_rank:
        conclusion = "本次必要页排名不变；没有证据表明背景文字带来收益。"
    else:
        conclusion = "本次必要页排名下降；不能把背景文字概括成所有问题都会改善。"
    print("结论：", conclusion)
    role = "main" if record["case_id"] == "contextual_cv_three_methods" else "check"
    result = {
        "case_id": record["case_id"],
        "method": "给片段补充模型生成的背景（Contextual Retrieval）",
        "role": role,
        "query": record["query"],
        "before": before,
        "after": after,
        "model_outputs": record.get("model_outputs", {}),
        "provenance": {
            "status": "fresh_api_run",
            "index_scope": "candidate_set_teaching_experiment",
            "candidate_selection": "BM25 top-12 per tutorial query; no expected_pages/reference_answer used for indexing",
            "raw_output_coverage": f"{len(raw_generated_contexts)}/{len(candidate_rows)} candidate chunks have raw model outputs",
            "search_text_rule": "validated_context + original_text; original_text remains separate",
        },
        "annotation_check_after_result": {
            "expected_pages": annotation["expected_pages"],
            "evidence_pages": sorted({
                int(span.get("page"))
                for claim in annotation["essential_evidence_spans"]
                if isinstance(claim, dict)
                for span in claim.get("spans", [])
                if isinstance(span, dict) and str(span.get("page", "")).isdigit()
            }),
        },
    }
    required_provenance = {"status", "index_scope", "candidate_selection", "raw_output_coverage"}
    assert required_provenance <= result["provenance"].keys()
    if role == "check":
        result["check_purpose"] = "说明不适用或限制"
    emit_tutorial_audit(result)

for record in records:
    record["annotation"] = load_annotation(record["case_id"])
    print(f"\n--- 上下文补充对照：{record['query']}（模型={RAG_LLM_MODEL}）---")
    print_contextual_report(record)



--- 上下文补充对照：南瓜书第二章的主题，究竟是评估模型优劣还是挑业务场景？（模型=glm-4-flash）---
问题： 南瓜书第二章的主题，究竟是评估模型优劣还是挑业务场景？
改动前：必要页排名 1 ，覆盖 [18] ；改动后：必要页排名 2 ，覆盖 [18]
模型补充的一句背景： 本段背景：解释周志华《机器学习》省略推导细节的原因，并介绍南瓜书作为辅助学习资料的使用方法。
资料量：候选 21 条；返回 8 → 8 条；字符数 2032 → 2424 ；背景生成调用 21 次
结论： 本次必要页排名下降；不能把背景文字概括成所有问题都会改善。



--- 上下文补充对照：第2.2节列出的三种模型评估办法分别叫什么？（模型=glm-4-flash）---
问题： 第2.2节列出的三种模型评估办法分别叫什么？
改动前：必要页排名 3 ，覆盖 [18] ；改动后：必要页排名 1 ，覆盖 [18]
模型补充的一句背景： 本段背景：解释周志华《机器学习》省略推导细节的原因，并介绍南瓜书作为辅助学习资料的使用方法。
资料量：候选 21 条；返回 8 → 8 条；字符数 2040 → 2430 ；背景生成调用 21 次
结论： 本次必要页排名改善；仍需在更多问题上验证。


## 结果解读

本节解读上方当前保存的真实 API 运行结果：第 2.2 节三种评估方法的问题中，必要页由第 3 名升到第 1；复查第二章主题的问题中，必要页由第 1 名降到第 2。只比较同一候选集内原片段与“校验后的背景 + 原片段”的排名；它不能外推为全库收益，也不能据两道题断言方法普遍有效。背景文字只用于建立索引，回答仍必须回到原始片段核对。


## CCH 和 Contextual Retrieval 不要混为一谈

CCH 为片段补上文档或章节标题；Contextual Retrieval 更进一步，让模型同时看到足够的章节内容和当前片段，再生成一两句“它位于哪里、在说明什么”的背景。这样可以补足代词、隐含引用和局部公式所缺的信息，而不只是重复章节名。

一个可检查的 Prompt 至少要说明：

1. 只描述片段在文档中的位置和作用，不回答用户问题；
2. 不添加原文没有的事实、数字或页码；
3. 控制长度，避免生成的背景盖过原片段；
4. 生成失败时立即报错并终止本次索引构建，修复 API 或数据问题后重跑，不静默回退。

背景只用于索引，回答阶段仍使用原始片段。每个 chunk 一次模型调用，成本与片段数成正比。当前案例明确是评测问题预筛后的候选集实验；若要做离线全库索引，应对全部 chunks 生成并缓存，不能把本页数字称为全库结果。代码把 raw_generated_contexts、validated_contexts、validation_notes 和 search_text 分开保存；校验只看原始 chunk、页文本的目录特征与长度规则，不使用 expected_pages、reference_answer 或目标排名。


## 生成片段背景文字的提示要点

本页已经用 `.env` 中配置的模型完成真实候选集实验；下面的片段只提炼同一实现的提示词与字段边界，不构成另一套实验结果。

```python
CONTEXT_PROMPT = """你在建立检索索引。
完整文档：
<document>
{document}
</document>
当前片段：
<chunk>
{chunk}
</chunk>
请用不超过两句话说明当前片段在文档中的位置和作用。
不要回答任何用户问题，不要编造文档没有的事实。
"""

def contextual_index_text(document: str, chunk: str, llm) -> str:
    context = llm.invoke(
        CONTEXT_PROMPT.format(document=document[:6000], chunk=chunk)
    ).content.strip()
    return f"{context}\n原始片段：{chunk}"

# 生产使用时把 raw_context、validated_context、validation_notes、original_text 分字段保存；
# 只有 validated_context + original_text 组成 search_text，回答仍引用 original_text。
```
